# galaxydisksize: a short demonstration

This notebook shows the reusable parts of the library on the committed
measurement tables: fitting the HI size-mass relation, computing size
residuals about the HI-to-optical baseline, and running the Kaplan-Meier
estimator for the beam-size upper limits.

It does not rebuild the manuscript; that is the job of the Snakemake workflow
described in the top-level `README.md`.

In [ ]:
from pathlib import Path

import numpy as np

import galaxydisksize as gds
from galaxydisksize.io import load_measurements

# Resolve the repository root whether the notebook runs from notebooks/ or root.
root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent
print("galaxydisksize", gds.__version__)

## 1. Fit the HI size-mass relation

`log10(D_HI) = m * log10(M_HI) + b`, with intrinsic scatter, on the isolated
AMIGA galaxies. A slope near 0.5 means a constant mean HI surface density.

In [ ]:
iso = load_measurements(root / "data" / "isolated_galaxies_results.csv")
detected = iso[(iso["hi_mass"] > 0) & (iso["hi_diameter_kpc"] > 0)]

fit = gds.fit_mass_size(
    np.log10(detected["hi_mass"]),
    np.log10(detected["hi_diameter_kpc"]),
    seed=42,
)
print(f"N        = {fit.n_data}")
print(f"slope    = {fit.slope:.3f}")
print(f"intercept= {fit.intercept:.3f}")
print(f"scatter  = {fit.scatter:.3f} dex")

## 2. Size residuals about the D_HI-D_25 baseline

In [ ]:
valid = detected[detected["optical_diameter_kpc"] > 0]
slope, intercept, scatter = gds.fit_baseline(
    np.log10(valid["optical_diameter_kpc"]),
    np.log10(valid["hi_diameter_kpc"]),
)
residual = gds.size_residual(
    np.log10(valid["hi_diameter_kpc"]),
    np.log10(valid["optical_diameter_kpc"]),
    slope,
    intercept,
)
print(f"baseline slope     = {slope:.3f}")
print(f"baseline scatter   = {scatter:.3f} dex")
print(f"median residual    = {np.median(residual):+.3f} dex")

## 3. Kaplan-Meier estimate with beam-size upper limits

The augmented HCG table flags non-detections and beam-unresolved members as
upper limits. The Kaplan-Meier estimator handles them as left-censored data.

In [ ]:
hcg = load_measurements(
    root / "data" / "interacting_galaxies_results_with_upperlimits_bmaj.csv",
    require_upper_limit_flag=True,
)
hcg = hcg[hcg["optical_diameter_kpc"] > 0]
delta = gds.size_residual(
    np.log10(hcg["hi_diameter_kpc"]),
    np.log10(hcg["optical_diameter_kpc"]),
    slope,
    intercept,
)
km = gds.kaplan_meier_left_censored(delta, hcg["is_upper_limit"].astype(bool))
print(f"KM median residual = {km.median:+.3f} dex (nan = unconstrained bound)")
print(f"fraction below baseline = {100 * km.fraction_below(0.0):.0f}%")